# Intensity demodulator phase calibration — filtered receiver INT2

This aligns the I quadrature with a **positive DC input** at the ZAD-1+ IF port. Turn off intensity feedback while doing this test; use your established safe DC level. LO remains DAC228/1 at 90 MHz, RF goes to ADC224/1, and the ADC mixer remains at the same 90 MHz. The optical setpoint, PI gains, carrier amplitude and RFDC phases are not configured by this notebook.

Load the matching bit/HWH beforehand. Downloading resets all loops. To attach in a new notebook without downloading, use the first cell. Both strong calibration filters now average I/Q before `atan2`. The old one’s-complement functions are unnecessary.


In [ ]:
from pynq import Overlay
import importlib
import numpy as np
import matplotlib.pyplot as plt
import intensity_phase_calibration as phase_cal
importlib.invalidate_caches()
phase_cal = importlib.reload(phase_cal)
# Edit to the matching loaded bit/HWH pair. download=False preserves running hardware.
# If ol already exists, omit the next line.
ol = Overlay("/home/xilinx/Bitfiles/FreqLocksBranch/overlays/TransportPhaseLockFF.bit", download=False)
print("Helper:", phase_cal.__file__)
assert ol.intensity_limits.read(8) == 0x494E5402


## Read, correct, verify
Confirm that the external DC source is present and feedback is disabled. The first cell below reads only. No zero reset of the correction is necessary. The second cell **adds** the measured residual to the correction already programmed, so repeating it does not replace the previous correction with the residual.


In [ ]:
before = phase_cal.read_phase(ol)
print("Measured phase [rad]:", before["phase_rad"])
plt.plot(before["phase_samples_rad"])
plt.ylabel("Demodulated phase [rad]"); plt.xlabel("30 kS/s sample"); plt.grid()


In [ ]:
correction_rad = phase_cal.correct_phase(ol, before["phase_rad"])
after = phase_cal.read_phase(ol)
print("Total programmed correction [rad]:", correction_rad)
print("Residual phase [rad]:", after["phase_rad"])
print("Circular concentration:", after["circular_concentration"])
plt.plot(after["phase_samples_rad"])
plt.ylabel("Corrected phase [rad]"); plt.xlabel("30 kS/s sample"); plt.grid()
assert abs(after["phase_rad"]) < 0.02, "Do not accept a large residual; inspect LO/DC/ADC settings."


Reconnect the optical sensor, leaving LO phase/amplitude, ADC mixer and cabling fixed. Restore the known working optical setpoint and PI settings. Verify the optical power at one existing calibration point before a full sweep. A zero phase reading with no input is not a calibration: use a nonzero DC source and confirm the I-channel response separately.
